# FMP 재무데이터 분석 v1 (US_IS/BS/CF_from_FMP)

`fmp_fs_analyzer_v1.py` 사용. DART 분석(`dart_fs_analyzer_v4`)과 같은 인터페이스.

**DART 와 다른 점**
- 단위 USD (`unit=1e6` → $M)
- FMP 분기값은 이미 분기 flow → 분기화 없음. `period`(회계분기) 대신 `date`(분기말)를 달력 분기로 변환해 정렬
- 저장 라이브러리(V4)가 단순 INSERT 라 같은 (ticker, date, item) 이 여러 번 쌓임 → **id 최대 행(최신 수집분)만 사용**
- 기업명 없음 (`name_table` 로 선택 매핑)

**데이터 업데이트**: `python US_FMP_FS_1_RUN_UPDATE.py` (2번 파일은 import 전용). 중복이 쌓이면 `python US_FMP_FS_2_DB_SAVE_LIB.py --truncate` 후 `--quarters 60` 재적재.

In [2]:
# ==========================================================
# PART 1: 환경 + Control Panel + DB
# ==========================================================
import sys
from pathlib import Path
import pandas as pd

def add_repo_path():
    for parent in [Path.cwd()] + list(Path.cwd().parents):
        if (parent / 'DATA').exists():
            if str(parent) not in sys.path:
                sys.path.insert(0, str(parent))
            return parent
    raise FileNotFoundError('DATA 폴더를 찾을 수 없습니다')

PROJECT_ROOT = add_repo_path()
sys.path.insert(0, str(Path.cwd()))

from DATA import config
import FMP_fs_analyzer_v1 as F

engine = config.get_engine(config.get_db_info())

# ---- Control Panel ----
ASOF        = None          # 기준 분기 '2026Q2'. None → 최근 완전 적재 분기 자동
START       = '2019-01-01'  # 패널 시작일 (TTM/평균용 여유 포함)
TICKERS     = None          # None=전체, 또는 ['AAPL','MSFT']
NAME_TABLE  = None          # 기업명 테이블 (예: 'us_ticker_master'); 없으면 None
TOP_N       = 30
UNIT        = 1e6           # USD → $M
PANEL_CACHE = Path('fmp_panel_cache.parquet')   # None 이면 캐시 안 씀. 데이터 재적재 후에는 삭제!

pd.set_option('display.width', 250)
pd.set_option('display.max_columns', 40)
pd.set_option('display.float_format', lambda x: f'{x:,.2f}')

## 0. 적재 상태 점검
`dup_factor` 가 2~3 이상이면 중복이 많이 쌓인 것 → `--truncate` 후 재적재 권장 (분석은 최신 행만 쓰므로 결과에는 영향 없음, 속도만 느려짐).

In [3]:
F.duplicate_report(engine)

,table,total_rows,unique_keys,dup_factor,n_ticker,max_date
0,US_IS_from_FMP,3670520,3239152,1.13,1989,2026-07-15
1,US_BS_from_FMP,5745652,5067260,1.13,1990,2026-07-15
2,US_CF_from_FMP,3921570,3459150,1.13,1989,2026-07-26


## 1. 적재된 항목 목록 (키워드 검색)
`mapped` = concept 매핑. 빈칸인 항목이 필요하면 `F.CONCEPTS` 에 추가.

In [4]:
F.search_items(engine, 'income')
# F.search_items(engine, sj_div='BS')
# F.search_items(engine)                       # 전체

,mapped,sj_div,item,n_ticker,n_rows,min_date,max_date
0,세전이익,IS,incomeBeforeTax,1989,131090,1996-03-31,2026-07-15
1,,IS,incomeBeforeTaxRatio,1989,131090,1996-03-31,2026-07-15
2,법인세,IS,incomeTaxExpense,1989,131090,1996-03-31,2026-07-15
3,,IS,interestIncome,1989,131090,1996-03-31,2026-07-15
4,당기순이익,IS,netIncome,1989,131090,1996-03-31,2026-07-15
5,,IS,netIncomeRatio,1989,131090,1996-03-31,2026-07-15
6,영업이익,IS,operatingIncome,1989,131090,1996-03-31,2026-07-15
7,,IS,operatingIncomeRatio,1989,131090,1996-03-31,2026-07-15
8,,IS,totalOtherIncomeExpensesNet,1989,131090,1996-03-31,2026-07-15
9,,BS,accumulatedOtherComprehensiveIncomeLoss,1990,130583,1987-12-31,2026-07-15


## 2. 패널 구축 (1회)

In [5]:
if PANEL_CACHE and PANEL_CACHE.exists() and TICKERS is None:
    panel = pd.read_parquet(PANEL_CACHE)
    panel['q'] = pd.PeriodIndex(panel['q'], freq='Q')
    print(f'cache load: {PANEL_CACHE} ({len(panel):,} rows)')
else:
    panel = F.build_panel(engine, tickers=TICKERS, start=START, name_table=NAME_TABLE)
    if PANEL_CACHE and TICKERS is None:
        panel.assign(q=panel['q'].astype(str)).to_parquet(PANEL_CACHE, index=False)
        print(f'cache save: {PANEL_CACHE}')
print(panel.shape, panel['ticker'].nunique(), 'tickers')
F.coverage_report(panel)

cache save: fmp_panel_cache.parquet
(2204476, 8) 1988 tickers


,n_ticker,n_rows,min_q,max_q
concept,,,,
현금성자산,1988,55120,2019Q1,2026Q3
유동부채,1988,55120,2019Q1,2026Q3
유형자산,1988,55120,2019Q1,2026Q3
자본,1988,55120,2019Q1,2026Q3
순차입금,1988,55120,2019Q1,2026Q3
비지배지분,1988,55120,2019Q1,2026Q3
부채,1988,55120,2019Q1,2026Q3
자산,1988,55120,2019Q1,2026Q3
장기차입금,1988,55120,2019Q1,2026Q3


## 3. 종목별 시계열
`ts.attrs['fiscal_dates']` 에 달력 분기 ↔ 실제 회계분기말 매핑.

In [6]:
ts = F.get_ts(panel, 'AAPL', ['매출액', '매출총이익', '영업이익', '당기순이익', 'EPS', '자본', '영업현금흐름', 'FCF'], start='2023Q1', unit=UNIT)
print(ts.attrs['ticker'], ts.attrs['fiscal_dates'].get(ts.index[-1]))
ts

AAPL 2026-06-27


concept,매출액,매출총이익,영업이익,당기순이익,EPS,자본,영업현금흐름,FCF
q,,,,,,,,
2023Q2,"94,836.00","41,976.00","28,318.00","24,160.00",1.52,"62,158.00","28,560.00","25,644.00"
2023Q3,"81,797.00","36,413.00","22,998.00","19,881.00",1.26,"60,274.00","26,380.00","24,287.00"
2023Q4,"119,575.00","54,855.00","40,373.00","33,916.00",2.18,"74,100.00","39,895.00","37,503.00"
2024Q1,"90,753.00","42,271.00","27,900.00","23,636.00",1.53,"74,194.00","22,690.00","20,694.00"
2024Q2,"85,777.00","39,678.00","25,352.00","21,448.00",1.40,"66,708.00","28,858.00","26,707.00"
2024Q3,"94,930.00","43,879.00","29,591.00","14,736.00",0.97,"56,950.00","26,811.00","23,903.00"
2024Q4,"124,300.00","58,275.00","42,832.00","36,330.00",2.40,"66,758.00","29,935.00","26,995.00"
2025Q1,"95,359.00","44,867.00","29,589.00","24,780.00",1.65,"66,796.00","23,952.00","20,881.00"
2025Q2,"94,036.00","43,718.00","28,202.00","23,434.00",1.57,"65,830.00","27,867.00","24,405.00"


## 4. YoY / QoQ 상위 N
`min_base` USD: 기준값 하한 (1e7 = $10M).

In [7]:
F.yoy_screen(panel, '매출액', n=TOP_N, asof=ASOF, unit=UNIT, min_base=1e7)

,ticker,company_name,base_q,t_q,매출액(t-4),매출액(t),growth_%
0,YPF,,2025Q1,2026Q1,"4,600.00","6,956,434.00","151,126.83"
1,QXO,,2025Q1,2026Q1,13.51,"1,730.20","12,708.71"
2,MDC,,2025Q1,2026Q1,24.77,"1,209.07","4,780.78"
3,NRZ,,2025Q1,2026Q1,28.89,375.06,"1,198.46"
4,GMAB,,2025Q1,2026Q1,105.23,899.32,754.59
5,ORC,,2025Q1,2026Q1,21.35,157.88,639.54
6,CYH,,2025Q1,2026Q1,"3,159.00","12,158.00",284.87
7,TRX,,2025Q1,2026Q1,13.15,46.51,253.58
8,UNIT,,2025Q1,2026Q1,293.91,987.50,235.99
9,RYN,,2025Q1,2026Q1,82.92,276.80,233.81


In [8]:
F.yoy_screen(panel, '영업이익', n=TOP_N, asof=ASOF, unit=UNIT, min_base=5e6)

,ticker,company_name,base_q,t_q,영업이익(t-4),영업이익(t),growth_%
0,YPF,,2025Q1,2026Q1,412.00,"1,241,859.00","301,322.09"
1,HOV,,2025Q1,2026Q1,19.55,562.66,"2,777.34"
2,TPX,,2025Q1,2026Q1,13.20,207.60,"1,472.73"
3,ALNY,,2025Q1,2026Q1,18.08,268.64,"1,386.07"
4,F,,2025Q1,2026Q1,164.00,"2,329.00","1,320.12"
5,ALB,,2025Q1,2026Q1,17.54,233.51,"1,231.43"
6,AGI,,2025Q1,2026Q1,25.70,337.66,"1,213.84"
7,ANDE,,2025Q1,2026Q1,7.12,90.72,"1,174.47"
8,DINO,,2025Q1,2026Q1,81.00,907.00,"1,019.75"
9,GAU,,2025Q1,2026Q1,8.10,86.14,963.24


In [ ]:
F.qoq_screen(panel, '매출액', n=TOP_N, asof=ASOF, unit=UNIT)

In [ ]:
F.qoq_screen(panel, '영업이익', n=TOP_N, asof=ASOF, unit=UNIT)

## 5. 영업이익 흑자전환

In [ ]:
F.turnaround_screen(panel, basis='yoy', asof=ASOF, unit=UNIT)

In [ ]:
F.turnaround_screen(panel, basis='qoq', asof=ASOF, unit=UNIT)

## 6. 재무비율 스크리너
`OPM, NPM, GPM, EBITDA_margin, ROE, ROE_지배, ROA, ROIC, 부채비율, 순차입금비율, OCF_margin, FCF_margin, FCF_conversion`

- 손익·CF TTM 합, BS 기초/기말 평균
- ROIC = OP×(1−유효세율)/(자본+순차입금) 평균. 순차입금은 FMP `netDebt` 우선, 없으면 totalDebt − 현금성자산
- 유효세율 = TTM 법인세/세전이익 (0~35% 클립, 산출 불가 시 21%)

In [ ]:
ratios = F.compute_ratios(panel, asof=ASOF, ttm=True, unit=UNIT)
ratios.shape

In [ ]:
F.ratio_screen(None, 'ROE',  n=TOP_N, ratios_df=ratios, unit=UNIT)

In [ ]:
F.ratio_screen(None, 'ROIC', n=TOP_N, ratios_df=ratios, unit=UNIT)

In [ ]:
F.ratio_screen(None, 'OPM',  n=TOP_N, ratios_df=ratios, unit=UNIT)

In [ ]:
F.ratio_screen(None, 'FCF_margin', n=TOP_N, ratios_df=ratios, unit=UNIT)

In [ ]:
F.ratio_screen(None, '부채비율', n=TOP_N, ratios_df=ratios, unit=UNIT, ascending=True)

In [ ]:
ratios[ratios['ticker'].isin(['AAPL', 'MSFT', 'NVDA'])].T